# CLAUD·IA — Primer fine-tuning supervisado

**Voz en español → texto en inglés · SeamlessM4T v2**

Este notebook piloto entrena dos proyecciones del último bloque de atención del decodificador. Los demás pesos permanecen congelados. Los 16 pares de entrenamiento sirven para realizar actualizaciones reales. Los 4 pares de validación permiten comparar la pérdida antes y después. Este pequeño conjunto ayuda a verificar el método y la memoria disponible.

[Documentación de SeamlessM4T v2](https://huggingface.co/docs/transformers/model_doc/seamless_m4t_v2) · [Datos FLEURS](https://huggingface.co/datasets/google/fleurs)

## 1. Preparar Colab

Selecciona **Runtime → Change runtime type → T4 GPU**. Ejecuta la instalación y reinicia el kernel si Colab lo solicita. Importa `fleurs_pilote.zip` desde el panel **Archivos** cuando el entorno GPU esté activo.

**¿Por qué instalar estos paquetes?** `%pip` instala las bibliotecas en el entorno Python utilizado por el notebook.

- **PyTorch (`torch`)** maneja tensores, envía los cálculos a la GPU, calcula gradientes y permite actualizar los pesos.
- **`transformers`** proporciona SeamlessM4T v2, su procesador y su código de generación. Se fija la versión 5.22.1 utilizada en nuestras pruebas.
- **`[sentencepiece]`** agrega las dependencias para segmentar texto. El tokenizador utiliza su vocabulario para representar frases en inglés con identificadores numéricos.
- **`huggingface-hub`** descarga los archivos del modelo preentrenado. El intervalo `>=1.32,<2` mantiene la compatibilidad con las bibliotecas del entorno, como Datasets y Diffusers.
- **`soundfile`** lee archivos WAV y proporciona las muestras numéricas y su frecuencia.

In [ ]:
%pip install "torch>=2.6" "transformers[sentencepiece]==5.22.1" "huggingface-hub>=1.32,<2" soundfile

## 2. Verificar la GPU y abrir el conjunto piloto

El notebook lee el archivo comprimido generado por el notebook 02. Comprueba las frecuencias de muestreo y la separación de las frases entre entrenamiento y validación.

**Del ZIP a la señal de audio**

1. `Path` representa la ruta de `fleurs_pilote.zip` y permite comprobar que el archivo existe.
2. `ZipFile` abre el archivo comprimido y lee sus archivos. Cada línea de un archivo JSONL contiene un ejemplo. `json.loads()` transforma la línea en un diccionario Python con el identificador, la ruta WAV y el texto inglés.
3. `io.BytesIO` convierte los bytes del WAV leídos del ZIP en un archivo virtual en memoria. Así, `soundfile.read()` puede leer el audio sin extraer el WAV al disco. `archive.read(...)` devuelve los bytes y `BytesIO(...)` ofrece a `soundfile` un objeto que permite leer y desplazarse.
4. `soundfile` devuelve la señal y su frecuencia. El código comprueba que el audio sea mono, tenga una frecuencia de 16 kHz y no supere ocho segundos.

`json`, `io`, `pathlib` y `zipfile` pertenecen a la biblioteca estándar de Python y no requieren una instalación adicional.

**Si la T4 no está seleccionada:** en nuestra prueba, `torch.cuda.is_available()` devolvió `False` porque el entorno seguía en modo CPU. El código se detuvo antes de abrir el ZIP o cargar el modelo. Después de seleccionar la T4, espera a que se conecte y vuelve a ejecutar esta celda.

In [ ]:
import json
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import soundfile as sf
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Selecciona una GPU T4 en Colab y vuelve a ejecutar esta celda.")

device = torch.device("cuda")
archive_path = Path("fleurs_pilote.zip")
sample_rate = 16_000
if not archive_path.is_file():
    raise FileNotFoundError("Importa fleurs_pilote.zip desde el panel Archivos.")

def read_split(archive, split):
    examples = []
    for line in archive.read(f"{split}.jsonl").decode("utf-8").splitlines():
        record = json.loads(line)
        audio, rate = sf.read(BytesIO(archive.read(record["audio_path"])), dtype="float32")
        if rate != sample_rate or audio.ndim != 1 or not 0 < len(audio) <= 8 * rate:
            raise ValueError(f"Audio inesperado para la frase {record['sentence_id']}.")
        examples.append({**record, "audio": audio})
    return examples

with ZipFile(archive_path) as archive:
    train_examples = read_split(archive, "train")
    validation_examples = read_split(archive, "validation")

train_ids = {example["sentence_id"] for example in train_examples}
validation_ids = {example["sentence_id"] for example in validation_examples}
if len(train_examples) != 16 or len(validation_examples) != 4:
    raise ValueError("El conjunto piloto debe contener 16 pares de entrenamiento y 4 de validación.")
if len(train_ids) != 16 or len(validation_ids) != 4 or train_ids & validation_ids:
    raise ValueError("Hay identificadores de frase repetidos.")

print(torch.cuda.get_device_name(0))
print(f"{len(train_examples)} pares de entrenamiento, {len(validation_examples)} validation.")

## 3. Cargar el modelo y seleccionar los pesos que se entrenarán

Cargamos la versión **voz → texto**. En la última capa del decodificador entrenamos las proyecciones de **consulta** (*query*) y **valor** (*value*) de la atención hacia el audio. La consulta ayuda al decodificador a localizar la información útil en la representación de audio; el valor transporta la información seleccionada.

Mantenemos los pesos congelados en FP16 y los pesos entrenables en FP32. Si la caché está vacía, se descargan los archivos del modelo.

`AutoProcessor` prepara las entradas de audio y texto. `SeamlessM4Tv2ForSpeechToText` carga el modelo que traduce audio a texto. `requires_grad_(False)` congela los parámetros y luego el código reactiva los gradientes solo en las dos proyecciones seleccionadas.

In [ ]:
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToText

model_id = "facebook/seamless-m4t-v2-large"
target_language = "eng"

processor = AutoProcessor.from_pretrained(model_id)
model = SeamlessM4Tv2ForSpeechToText.from_pretrained(
    model_id, dtype=torch.float16, use_safetensors=True
).to(device)
model.requires_grad_(False)
model.eval()

last_layer = model.text_decoder.layers[-1]
for projection in (last_layer.cross_attention.q_proj, last_layer.cross_attention.v_proj):
    projection.float().requires_grad_(True)

trainable = {name: parameter for name, parameter in model.named_parameters() if parameter.requires_grad}
initial_weights = {name: parameter.detach().cpu().clone() for name, parameter in trainable.items()}
print(f"Parámetros entrenables: {sum(parameter.numel() for parameter in trainable.values()):,}")
print("\n".join(trainable))

## 4. Medir antes del fine-tuning

Para calcular la pérdida, el decodificador recibe el comienzo correcto de la frase y debe predecir cada unidad siguiente. El texto inglés de referencia sirve para calcular la pérdida. Para la traducción mostrada, el modelo recibe únicamente el audio y el código de idioma. Mantenemos las mismas condiciones antes y después. `statistics.mean()` calcula el promedio de pérdida de los cuatro ejemplos de validación.

In [ ]:
from statistics import mean

language_id = model.generation_config.text_decoder_lang_to_code_id[target_language]

def audio_inputs(example):
    inputs = processor(
        audio=example["audio"], sampling_rate=sample_rate, return_tensors="pt"
    ).to(device)
    inputs["input_features"] = inputs["input_features"].to(dtype=torch.float16)
    return inputs

def supervised_loss(example):
    inputs = audio_inputs(example)
    token_ids = processor.tokenizer(example["target_text"], add_special_tokens=False).input_ids
    decoder_ids = torch.tensor(
        [[model.config.decoder_start_token_id, language_id, *token_ids]], device=device
    )
    labels = torch.tensor([[-100, *token_ids, model.config.eos_token_id]], device=device)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        return model(**inputs, decoder_input_ids=decoder_ids, labels=labels, use_cache=False).loss

@torch.no_grad()
def validation_loss():
    model.eval()
    return mean(supervised_loss(example).item() for example in validation_examples)

@torch.no_grad()
def translate(example):
    model.eval()
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        tokens = model.generate(
            **audio_inputs(example), tgt_lang=target_language, max_new_tokens=80,
            return_dict_in_generate=False,
        )
    return processor.batch_decode(
        tokens.cpu(), skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

validation_before = validation_loss()
example = validation_examples[0]
prediction_before = translate(example)
print(f"Pérdida media de validación antes: {validation_before:.4f}")
print(f"Referencia: {example['target_text']}")
print(f"Predicción antes: {prediction_before}")

## 5. Entrenar y comparar

Una época recorre los 16 pares una vez, con lotes de una frase. La retropropagación calcula los gradientes y AdamW ajusta los pesos seleccionados. El escalado de gradientes facilita el entrenamiento con precisión mixta. El modo de evaluación desactiva la aleatoriedad de ciertas capas durante este piloto, pero permite la retropropagación.

`random.Random(42)` permite reproducir el orden de mezcla de los ejemplos. `math.isfinite()` comprueba que la pérdida sea un número finito antes de continuar.

Si ejecutas de nuevo esta celda, se restauran los pesos preentrenados. La variación de pérdida de solo cuatro frases es orientativa.

In [ ]:
import math
import random

learning_rate = 3e-5
rng = random.Random(42)
order = train_examples.copy()
rng.shuffle(order)

with torch.no_grad():
    for name, parameter in trainable.items():
        parameter.copy_(initial_weights[name])

optimizer = torch.optim.AdamW(trainable.values(), lr=learning_rate, weight_decay=0)
scaler = torch.amp.GradScaler("cuda", init_scale=1024)
train_losses = []

for step, example in enumerate(order, start=1):
    optimizer.zero_grad(set_to_none=True)
    loss = supervised_loss(example)
    loss_value = loss.item()
    if not math.isfinite(loss_value):
        raise RuntimeError(f"Pérdida no finita en el paso {step}.")
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    train_losses.append(loss_value)
    print(f"Paso {step:02d}/{len(order)} — pérdida: {loss_value:.4f}")

updated_names = [
    name for name, parameter in trainable.items()
    if not torch.equal(parameter.detach().cpu(), initial_weights[name])
]
if not updated_names:
    raise RuntimeError("Ningún peso entrenable cambió durante la prueba piloto.")

validation_after = validation_loss()
example = validation_examples[0]
prediction_after = translate(example)
print(f"Parámetros modificados: {len(updated_names)} tensores")
print(f"Pérdida de validación antes: {validation_before:.4f}")
print(f"Pérdida de validación después: {validation_after:.4f}")
print(f"Referencia: {example['target_text']}")
print(f"Predicción antes: {prediction_before}")
print(f"Predicción después: {prediction_after}")

## 6. Guardar el resultado

El archivo comprimido contiene únicamente los pesos ajustados y el informe de la prueba. `safetensors` guarda los tensores modificados y `shutil.make_archive()` reúne el archivo de pesos y el informe JSON en un ZIP. Este archivo permitirá repetir la comparación y reincorporar los pesos al modelo vocal. Descárgalo antes de cerrar la sesión de Colab.

In [ ]:
import shutil
from safetensors.torch import save_file

output_dir = Path("fine_tuning_pilote")
output_dir.mkdir(exist_ok=True)
save_file(
    {name: parameter.detach().cpu().contiguous() for name, parameter in trainable.items()},
    output_dir / "poids_ajustes.safetensors",
)
report = {
    "base_model": model_id,
    "gpu": torch.cuda.get_device_name(0),
    "source_language": "spa",
    "target_language": target_language,
    "learning_rate": learning_rate,
    "train_sentence_ids": sorted(train_ids),
    "validation_sentence_ids": sorted(validation_ids),
    "training_losses": train_losses,
    "validation_loss_before": validation_before,
    "validation_loss_after": validation_after,
    "validation_example_id": example["sentence_id"],
    "reference": example["target_text"],
    "prediction_before": prediction_before,
    "prediction_after": prediction_after,
    "updated_parameters": updated_names,
}
(output_dir / "rapport.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
archive = shutil.make_archive(output_dir.name, "zip", root_dir=output_dir)
print(f"Archivo comprimido listo: {Path(archive).name}")